# Data Processing

This notebook takes the raw MultiScope dataset and turns it into a clean, de-identified starting point for the rest of the thesis.

**The data in one paragraph.** About 300 patients from Unidade Local de Saúde de Gaia e Espinho had their heart sounds (PCG) and electrocardiogram (ECG) recorded at the same time, at the four classic auscultation sites: aortic, pulmonary, tricuspid and mitral. Each patient also had an echocardiogram, and its measurements and written report are stored in a clinical spreadsheet. The recordings were made with two different apps that save the signals in different formats, so part of the work here is simply bringing both into the same shape:

| Patients | App | ECG | PCG |
|---|---|---|---|
| 1 - 108, plus 118, 119, 121, 122 | Rijuven | `.raw` (500 Hz) | `.mp3` |
| 109 onwards | New app (Multiscope) | `.csv` with header | `.csv` with header |

**What this notebook does, step by step**

1. Finds every recording and pairs each ECG with its PCG.
2. Loads the clinical spreadsheet and removes everything that identifies a person (name, health number, birthday).
3. Builds the global dataset: one table with a row per patient and one with a row per recording.

Signal embeddings from pretrained models, tabular foundation models and clustering will be added as the next sections.

Nothing here filters, resamples or otherwise changes the signals.

## Table of contents

1. Imports
2. Configuration
3. Helpers
4. Readers
5. Recording index
6. Clinical data
7. De-identification
8. Global dataset
9. Next steps

## 1. Imports

In [ ]:
# Standard library
import re
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

# Third party
import numpy as np
import pandas as pd
import soundfile as sf

## 2. Configuration

The dataset folder was reorganized once with **`scripts/organize_data.py`**, so this notebook expects the following layout. If you start from a fresh copy of the data, run `python scripts/organize_data.py --dry-run` to see the plan, and then run it without the flag.

```
data/DatasetCHVNGE/
├── New_app_patients/        one folder per patient (109+), ECG and PCG CSVs
├── Rijuven_patients/        Rijuven .raw (ECG) and .mp3 (PCG) files
├── DB_Multiscope/           clinical database, every version (copies found in patient folders start with 'from_<ID>_')
├── Patients_Multiscope/     patient list (contains names, used only for de-identification)
├── README/                  notes about the data
├── Duplicates/              loose copies of files that already exist in a patient folder
└── ...                      12-lead ECG folders, quality annotations
```

In [ ]:
# Project root: first folder, from the notebook location upwards, that contains a "data" folder
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir())

DATA_ROOT = PROJECT_ROOT / "data" / "DatasetCHVNGE"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"   # de-identified tables are saved here

NEW_APP_DIR = DATA_ROOT / "New_app_patients"
RIJUVEN_DIR = DATA_ROOT / "Rijuven_patients"
DB_DIR = DATA_ROOT / "DB_Multiscope"
PATIENTS_DIR = DATA_ROOT / "Patients_Multiscope"

CLINICAL_DB_GLOB = "21.07.2026_DB MultiScope*.xlsx"   # latest version of the clinical database
PATIENT_LIST_GLOB = "Patients_MultiScope*.xlsx"
REPORT_COLUMN = "Report_ecoTT"                        # free-text echocardiography report
IDENTIFIER_COLUMNS = ["NAME", "NSC", "PROC", "Birthday"]

RIJUVEN_ECG_FS = 500   # not stored in the .raw files, value from the original script

for folder in (NEW_APP_DIR, RIJUVEN_DIR, DB_DIR, PATIENTS_DIR):
    assert folder.is_dir(), f"Missing folder: {folder} (run scripts/organize_data.py first)"
print("Dataset root:", DATA_ROOT)

## 3. Helpers

In [ ]:
# Every valve spelling used by both apps, mapped to one code
SITE_MAP = {
    "av": "AV", "aortica": "AV",
    "mv": "MV", "mitral": "MV",
    "pv": "PV", "pulmonar": "PV",
    "tv": "TV", "tricuspide": "TV",
}


def strip_accents(text: str) -> str:
    """'Tricúspide' -> 'Tricuspide'."""
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")


def normalize_site(name) -> str | None:
    """Common valve code, None for empty/NaN, or the original name if unknown."""
    clean = strip_accents(str(name)).strip().lower()
    if clean in ("", "nan", "none"):
        return None
    return SITE_MAP.get(clean, str(name))


def read_text(path: Path) -> str:
    """Read a text file as UTF-8, falling back to Latin-1 for older exports."""
    try:
        return path.read_text(encoding="utf-8-sig")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")


def parse_number(value: str | None) -> float | None:
    """'500Hz' -> 500.0"""
    match = re.search(r"\d+(?:\.\d+)?", value or "")
    return float(match.group()) if match else None


def to_date(values: pd.Series) -> pd.Series:
    """Parse dates that come in mixed formats (Excel dates and 'dd/mm/yyyy' strings)."""
    return pd.to_datetime(values, errors="coerce", dayfirst=True, format="mixed")


def save_table(df: pd.DataFrame, name: str) -> None:
    """Save a table as parquet; mixed-type text columns are stored as strings."""
    df = df.copy()
    text_cols = [c for c in df.columns if df[c].dtype == object]
    df[text_cols] = df[text_cols].astype("string")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    print(f"Saved {OUTPUT_DIR / name}.parquet  ({len(df)} rows)")

## 4. Readers

One reader per format. Each returns a `Signal` with the samples, the sampling rate and any extra information from the file.

In [ ]:
@dataclass
class Signal:
    data: np.ndarray
    fs: float
    meta: dict = field(default_factory=dict)

    @property
    def duration(self) -> float:
        return len(self.data) / self.fs

In [ ]:
# New app: header lines 'key=value', then one packet per row starting with a timestamp
TIMESTAMP_RE = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}")


def read_new_app_csv(path: Path) -> Signal:
    header, packet_times, packets = {}, [], []
    for line in read_text(path).splitlines():
        line = line.strip().rstrip(",")
        if TIMESTAMP_RE.match(line):
            stamp, *values = line.split(",")
            packet_times.append(stamp)
            packets.append(np.array([v for v in values if v.strip()], dtype=float))
        elif "=" in line:
            key, value = line.split("=", 1)
            header[key.strip()] = value.strip()

    fs = parse_number(header.get("Sampling_frequency"))
    if fs is None:
        raise ValueError(f"No Sampling_frequency in header: {path.name}")
    data = np.concatenate(packets) if packets else np.empty(0)
    return Signal(data, fs, {"header": header, "packet_times": packet_times})

In [ ]:
# Rijuven: ECG as comma-separated text, PCG as mp3
def read_rijuven_ecg(path: Path) -> Signal:
    values = [v for v in re.split(r"[,\s]+", read_text(path).strip()) if v]
    return Signal(np.array(values, dtype=float), RIJUVEN_ECG_FS)


def read_rijuven_pcg(path: Path) -> Signal:
    data, fs = sf.read(path, always_2d=True)   # native sampling rate
    return Signal(data.mean(axis=1), float(fs))  # stereo -> mono

In [ ]:
def load_recording(row: pd.Series) -> tuple[Signal, Signal]:
    """Return (ecg, pcg) for one row of the recording index."""
    ecg_path, pcg_path = DATA_ROOT / row["ecg_path"], DATA_ROOT / row["pcg_path"]
    if row["source"] == "rijuven":
        return read_rijuven_ecg(ecg_path), read_rijuven_pcg(pcg_path)
    return read_new_app_csv(ecg_path), read_new_app_csv(pcg_path)

## 5. Recording index

One row per recording, with the ECG and PCG paths (relative to the dataset root). Files are paired by name: Rijuven by the file stem (`1_AV.raw` + `1_AV.mp3`), the new app by the timestamp in the name (ECG + PCG/STE).

In [ ]:
RIJUVEN_RE = re.compile(r"^(?P<patient>\d+)_(?P<site>[A-Za-z]+).*\.(?P<ext>raw|mp3)$", re.IGNORECASE)
NEW_APP_RE = re.compile(r"^[^_]+_(?P<valve>.+?)_(?P<signal>ECG|PCG|STE)_(?P<stamp>.+)\.csv$", re.IGNORECASE)


def index_rijuven() -> tuple[pd.DataFrame, list[str]]:
    records, unmatched = {}, []
    for path in sorted(RIJUVEN_DIR.iterdir()):
        m = RIJUVEN_RE.match(path.name)
        if not m:
            unmatched.append(path.name)
            continue
        rec = records.setdefault(path.stem, {
            "patient": int(m["patient"]), "source": "rijuven",
            "site": normalize_site(m["site"]), "recording_id": path.stem,
        })
        kind = "ecg" if m["ext"].lower() == "raw" else "pcg"
        rec[f"{kind}_path"] = str(path.relative_to(DATA_ROOT))
    return pd.DataFrame(records.values()), unmatched


def index_new_app() -> tuple[pd.DataFrame, list[str]]:
    records, unmatched = {}, []
    for path in sorted(NEW_APP_DIR.glob("*/*.csv")):
        m = NEW_APP_RE.match(path.name)
        if not m or not path.parent.name.isdigit():
            unmatched.append(str(path.relative_to(NEW_APP_DIR)))
            continue
        patient = int(path.parent.name)
        rec = records.setdefault((patient, m["stamp"]), {
            "patient": patient, "source": "new_app",
            "site": None, "recording_id": f"{patient}_{m['stamp']}",
        })
        kind = "ecg" if m["signal"].upper() == "ECG" else "pcg"
        rec[f"{kind}_path"] = str(path.relative_to(DATA_ROOT))
        rec["site"] = rec["site"] or normalize_site(m["valve"])   # the valve is often only in one of the two files
    return pd.DataFrame(records.values()), unmatched

In [ ]:
rijuven_index, rijuven_unmatched = index_rijuven()
new_app_index, new_app_unmatched = index_new_app()

recordings = (
    pd.concat([rijuven_index, new_app_index], ignore_index=True)
    .sort_values(["patient", "source", "recording_id"])
    .reset_index(drop=True)
)

print(f"Recordings: {len(recordings)}  (Rijuven: {len(rijuven_index)}, new app: {len(new_app_index)})")
print(f"Patients:   {recordings['patient'].nunique()}")
print(f"Missing ECG: {recordings['ecg_path'].isna().sum()}  |  Missing PCG: {recordings['pcg_path'].isna().sum()}")
print(f"Files not recognized: Rijuven {rijuven_unmatched}, new app {new_app_unmatched}")

pd.crosstab(recordings["site"].fillna("unknown"), recordings["source"], margins=True)

In [ ]:
# Patients with more than one recording at the same site (repeated measurements, see README)
repeats = recordings.groupby(["patient", "site"]).size()
repeats[repeats > 1].unstack(fill_value=0)

In [ ]:
# Sanity check: read the first recording of each app
for source, group in recordings.groupby("source"):
    ecg, pcg = load_recording(group.iloc[0])
    print(f"{source:8s} | ECG {ecg.fs:g} Hz, {ecg.duration:.1f} s | PCG {pcg.fs:g} Hz, {pcg.duration:.1f} s")

## 6. Clinical data

The latest version of the clinical database (one row per patient, echocardiography measurements and the report text) and the patient list. The patient list is only needed to know each patient's name, so it can be removed from the reports in the next section.

In [ ]:
def read_single(folder: Path, pattern: str, **kwargs) -> pd.DataFrame:
    """Read the only spreadsheet in a folder that matches a pattern."""
    matches = sorted(folder.glob(pattern))
    assert len(matches) == 1, f"Expected 1 file for '{pattern}', found {[m.name for m in matches]}"
    print("Reading", matches[0].name)
    return pd.read_excel(matches[0], **kwargs)


def clean_ids(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows with a numeric ID, stored as int."""
    df = df.assign(ID=pd.to_numeric(df["ID"], errors="coerce"))
    return df.dropna(subset=["ID"]).astype({"ID": int})


clinical_raw = clean_ids(read_single(DB_DIR, CLINICAL_DB_GLOB))
patient_list = clean_ids(read_single(PATIENTS_DIR, PATIENT_LIST_GLOB, header=1))   # real header is on the 2nd row

print(f"Clinical DB: {clinical_raw.shape[0]} rows x {clinical_raw.shape[1]} columns")
print(f"Duplicated IDs in the clinical DB: {clinical_raw['ID'].duplicated().sum()}")
print("Columns:", list(clinical_raw.columns))

## 7. De-identification

Removes the identifier columns, replaces the birthday with the age at the evaluation date, and replaces the patient's name with `[NAME]` wherever it appears in the report text. Only name parts with 4 or more letters are replaced, so short words such as "da" or "dos" stay intact.

After this section the tables with identifiers are deleted from memory, and nothing below uses them.

In [ ]:
def remove_name(text, name):
    """Replace every part of the patient's name in a report with [NAME]."""
    if not isinstance(text, str) or not isinstance(name, str):
        return text
    for part in set(name.split()):
        if len(part) >= 4:
            text = re.sub(rf"\b{re.escape(part)}\b", "[NAME]", text, flags=re.IGNORECASE)
    return text


def deidentify(clinical: pd.DataFrame, names: pd.Series) -> pd.DataFrame:
    out = clinical.copy()
    out["Age"] = ((to_date(out["EvaluationDate"]) - to_date(out["Birthday"])).dt.days / 365.25).round(1)
    out[REPORT_COLUMN] = [remove_name(text, names.get(pid)) for text, pid in zip(out[REPORT_COLUMN], out["ID"])]
    return out.drop(columns=IDENTIFIER_COLUMNS, errors="ignore")

In [ ]:
names = patient_list.drop_duplicates("ID").set_index("ID")["NAME"]
clinical = deidentify(clinical_raw, names)

has_report = clinical_raw[REPORT_COLUMN].notna()
changed = (clinical[REPORT_COLUMN] != clinical_raw[REPORT_COLUMN]) & has_report
no_name = has_report & ~clinical_raw["ID"].isin(names.index)
print(f"Reports: {has_report.sum()}  |  name removed in {changed.sum()}")
print(f"Reports of patients missing from the patient list (check by hand): {sorted(clinical_raw.loc[no_name, 'ID'])}")
print("Identifier columns left:", [c for c in IDENTIFIER_COLUMNS if c in clinical.columns])

del clinical_raw, patient_list, names   # identifiers stay out of the rest of the notebook

## 8. Global dataset

Two tables, saved in `data/processed/`:

- **`patients`**: one row per patient, the de-identified clinical data plus a summary of their recordings.
- **`recordings`**: one row per recording, with the ECG and PCG paths.

The signals themselves stay on disk and are loaded only when needed. Signal embeddings will be added later, linked by `recording_id`.

In [ ]:
per_patient = (
    recordings.groupby("patient")
    .agg(
        n_recordings=("recording_id", "size"),
        sites=("site", lambda s: ",".join(sorted(s.dropna().unique()))),
        apps=("source", lambda s: ",".join(sorted(s.unique()))),
    )
    .rename_axis("ID")
    .reset_index()
)

patients = clinical.merge(per_patient, on="ID", how="outer", indicator=True)
patients["has_clinical"] = patients["_merge"] != "right_only"
patients["has_signals"] = patients["_merge"] != "left_only"
patients["has_report"] = patients[REPORT_COLUMN].fillna("").astype(str).str.strip().ne("")
patients = patients.drop(columns="_merge").sort_values("ID").reset_index(drop=True)

pd.crosstab(patients["has_clinical"], patients["has_signals"], margins=True)

In [ ]:
print(f"Patients with signals, clinical data and a report: {(patients['has_signals'] & patients['has_report']).sum()}")
print("Signals but no clinical data:", patients.loc[~patients["has_clinical"], "ID"].tolist())
print("Clinical data but no signals:", patients.loc[~patients["has_signals"], "ID"].tolist())

In [ ]:
save_table(patients, "patients")
save_table(recordings, "recordings")

## 9. Next steps

- **Signal embeddings:** pass each PCG and ECG through a pretrained model and keep the output vector as a compact description of the recording.
- **Tabular foundation model:** use TabPFN on the clinical table as a baseline (for example, predicting pulmonary hypertension).
- **Clustering:** look at whether patients group together using the embeddings, coloring by app, valve and diagnosis.